# D1 · Satellite images and spectral bands: the Melamchi valley from space

> **Time:** about 2.5 hours  
> **You need:** Parts A, B and C, especially reading a raster with rasterio (C1), plotting with `extent=` and a percentile stretch (C2), rioxarray and `clip_box` (C3), and masks and `NaN` (C5).  
> **By the end you can:**
> 1. explain what a passive optical satellite such as **Sentinel-2** measures: reflected sunlight in **bands** of wavelength, from visible light to the short-wave infrared,
> 2. open a multi-band image with rasterio and rioxarray, give the bands their names, and turn stored numbers into **reflectance**,
> 3. make **true-colour** and **false-colour** composites with a stretch, read **spectral profiles**, and take a first look at the valley before and after the 2021 flood.

In Part C every raster had one band: elevation. A satellite image has **many bands** on the same grid, one for each kind of light. Together they show far more than a photograph: healthy forest, bare gravel, water and fresh river sediment each reflect light in their own way.

Part D works with two Sentinel-2 images of the **Melamchi valley** in Sindhupalchok district: one from 29 October 2020 and one from 11 November 2021. In **June 2021** a debris flow and flood swept down the Melamchi river and buried parts of the valley floor under sediment. This lesson teaches you to read the images; D2 and D3 measure what changed.

The images were downloaded once by `../data/get_sentinel2_melamchi.py`. It searched a **STAC catalogue** (an online index of satellite scenes) for clear scenes, cut out the valley, and saved two small GeoTIFFs. You can open that script to see how; the lessons themselves need no internet.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.* This credit line must appear wherever the images are shown.

**Counting lines:** lines that only set up or show a plot (`fig, ax = plt.subplots()`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 30 of 32 · [Course home](../README.md)  
← Previous: [C8 · Interpolation: a rainfall map from 24 gauges](../C_Raster/C08_Interpolation.ipynb) · Next: [D2 · Spectral indices: vegetation, water and bare ground in the Melamchi valley](D02_Spectral_indices.ipynb) →

## 1. What a satellite measures

The Sun sends out light of many **wavelengths**. Wavelength is measured in **nanometres** (nm): 1 nm is one billionth of a metre. Together, all wavelengths of light and radiation make up the **electromagnetic spectrum**.

- **Visible light**, roughly 400 to 700 nm: blue, green and red. Our eyes see only this part.
- **Near-infrared (NIR)**, from about 700 nm to about 1,400 nm. Healthy leaves reflect it strongly.
- **Short-wave infrared (SWIR)**, from about 1,400 nm to about 3,000 nm. It responds to water in leaves and soil, and to minerals.

The exact boundaries differ a little between textbooks. Sentinel-2's SWIR bands lie at 1,610 and 2,190 nm.

Sunlight hits the ground, and part of it is **reflected** back up. **Reflectance** is the fraction reflected: 0 means nothing, 1 means everything. A **passive** sensor measures this reflected sunlight. It needs daylight and cannot see through cloud. (**Active** sensors such as radar send their own signal, and see through cloud.)

**Sentinel-2** is the optical satellite mission of the European Union's Copernicus programme, run by the European Space Agency (ESA). In 2020 and 2021 it had two identical satellites, **Sentinel-2A** and **Sentinel-2B**, on the same orbit. Together they passed over each place about **every 5 days**. Each image has **13 bands**: four at **10 m**, six at **20 m** and three at **60 m** (those three are for measuring the atmosphere). **Level-2A** products are corrected for the atmosphere: they give **surface reflectance**, the light reflected by the ground itself.

Our files hold the 10 bands of 10 m and 20 m. The example lists them in a table (A9).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import rioxarray as rxr

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

bands = pd.DataFrame({
    "band": ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"],
    "name": ["blue", "green", "red", "red edge 1", "red edge 2", "red edge 3", "near-infrared", "narrow NIR", "SWIR 1", "SWIR 2"],
    "centre_nm": [490, 560, 665, 705, 740, 783, 842, 865, 1610, 2190],
    "native_m": [10, 10, 10, 20, 20, 20, 10, 20, 20, 20],
})
bands

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.axvspan(400, 700, color="gold", alpha=0.3, label="visible light")
ax.scatter(bands["centre_nm"], [0] * len(bands), color="black", zorder=3)
for i, (name, nm) in enumerate(zip(bands["band"], bands["centre_nm"])):
    above = i % 2 == 0                      # labels alternate above and below, so they do not overlap
    ax.annotate(name, (nm, 0), xytext=(0, 8 if above else -8), textcoords="offset points",
                ha="center", va="bottom" if above else "top", rotation=90)
ax.set_xlim(350, 2400)
ax.set_yticks([])
ax.set_xlabel("Wavelength (nm)")
ax.set_title("The 10 Sentinel-2 bands in our files")
ax.legend(loc="upper right")
plt.show()

The table has one row per band. `centre_nm` is the **central wavelength** of each band: a band is a narrow range of wavelengths around that centre. `native_m` is the cell size the satellite measures that band at.

On the strip, three bands sit inside the yellow visible range. Three **red-edge** bands follow, between red and NIR, where the reflectance of plants rises steeply. Two NIR bands come next, and the two SWIR bands lie far out at 1,610 and 2,190 nm.

In our files **all 10 bands were resampled to one 20 m grid** in UTM 45N, so they can be combined cell by cell (C4). The 10 m detail of four bands is lost, but the grid is simple.

### Practice

**Q1 · one line:** Show only the rows of `bands` whose `native_m` is 10.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bands[bands["native_m"] == 10]
```

B02, B03, B04 and B08: blue, green, red and NIR are measured at 10 m. Boolean filtering works on this table as on any other (A9).
</details>

**Q2 · two lines:** Add a column `centre_um` with the central wavelength in **micrometres** (1 µm = 1,000 nm), and show the columns `band` and `centre_um`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bands["centre_um"] = bands["centre_nm"] / 1000
bands[["band", "centre_um"]]
```

B02 lies at 0.49 µm and B12 at 2.19 µm. Many textbooks write wavelengths in µm; both units say the same.
</details>

**Q3 · three lines:** Read `s2_melamchi_scenes.csv` from `DATA` with pandas and print the columns `image`, `date` and `box_not_clear_pct`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
scenes = pd.read_csv(DATA / "s2_melamchi_scenes.csv")
print(scenes[["image", "date", "box_not_clear_pct"]])
```

The before image is from 2020-10-29 and the after image from 2021-11-11. Only 0.1% and 0.4% of the box was not clear ground (mostly cloud) on those days. The download script chose the clearest day in each autumn.
</details>

## 2. Opening a multi-band image with rasterio

You open a satellite image exactly like the DEM in C1: `with rasterio.open(path) as src:`. A few properties matter more now:

- `src.count`: the number of bands.
- `src.descriptions`: the band names stored in the file, if the maker saved them.
- `src.scales` and `src.offsets`: how to turn stored numbers into real values, one per band: **value × scale + offset**.
- `src.dtypes`, `src.nodata`, `src.crs`, `src.res` and `src.bounds`, as in C1.

`src.read(1)` reads **one** band as a 2-D array (rows, columns). `src.read()` reads **all** bands as a 3-D array (bands, rows, columns). `src.read([3, 2, 1])` reads a chosen list of bands. Band numbers in rasterio start at **1**, not 0.

In [ ]:
before_path = DATA / "s2_melamchi_before.tif"
after_path = DATA / "s2_melamchi_after.tif"

with rasterio.open(before_path) as src:
    print("bands:", src.count)
    print("names:", src.descriptions)
    print("scales:", src.scales)
    print("offsets:", src.offsets)
    print(src.dtypes[0], "| nodata:", src.nodata, "|", src.crs, "| cell size:", src.res)
    print(src.bounds)
    red_dn = src.read(3)          # band 3 = B04, red
    all_dn = src.read()           # all ten bands

print(red_dn.shape, red_dn.dtype)
print(all_dn.shape)

The file has 10 bands, named B02 to B12 in the order of the table in step 1. Each band has a scale of 0.0001 and an offset of 0.

The values are `uint16`: **unsigned** 16-bit whole numbers, from 0 to 65,535, with no negatives. Nodata is 0. The CRS is UTM 45N (EPSG:32645) with 20 m cells, the same CRS as the Part C grid (C4).

The bounds run from 350,260 to 364,300 m east and from 3,075,800 to 3,095,920 m north: about 14 km × 20 km.

`read(3)` gives one band of 1,006 rows × 702 columns. `read()` gives all of them at once: shape (10, 1006, 702), bands first.

### Practice

**Q1 · one line:** Print the number of rows and columns of the image from `src.height` and `src.width`. Open the file again in a `with` block.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(before_path) as src:
    print(src.height, src.width)
```

1,006 rows and 702 columns: the same numbers as the shape of `red_dn`.
</details>

**Q2 · two lines:** Read the blue, green and red bands of the **after** image in one call with `src.read([1, 2, 3])` and print the shape and dtype of the result.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(after_path) as src:
    rgb_dn = src.read([1, 2, 3])
print(rgb_dn.shape, rgb_dn.dtype)
```

(3, 1006, 702) and `uint16`: three bands, in the order you asked for. The bands of the after image are the same as those of the before image, on the same grid.
</details>

**Q3 · three lines:** Use `all_dn` to check that the bands of the before image are complete: for each band, count the cells equal to the nodata value 0. Loop over the band index with `range(10)` (A3).

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for i in range(10):
    print(i, (all_dn[i] == 0).sum())
```

0 nodata cells in every band of the before image. Index 0 of the NumPy array is band 1 of the file: NumPy counts from 0, rasterio from 1.
</details>

## 3. Bands with names: rioxarray

rioxarray (C2 to C4) opens the image as a 3-D DataArray with the dimensions **(band, y, x)**. The band dimension has coordinates 1 to 10.

Band numbers are confusing. B08 is the 7th band, not the 8th! It is safer to give the bands their **names**. rioxarray keeps the file's band descriptions in `.attrs["long_name"]`, and `.assign_coords(band=...)` replaces the numbers with those names.

Then `.sel(band="B04")` picks the red band by name, and `.sel(band=["B04", "B03", "B02"])` picks several. `masked=True` turns nodata (0) into `NaN`, as in C4, and makes the values `float32`.

In [ ]:
before_dn = rxr.open_rasterio(before_path, masked=True)
print(before_dn.dims, before_dn.shape, before_dn.dtype)
print("band coordinates:", before_dn.band.values)
print("names in the file:", before_dn.attrs["long_name"])

before_dn = before_dn.assign_coords(band=list(before_dn.attrs["long_name"]))
print("band coordinates now:", before_dn.band.values)

red = before_dn.sel(band="B04")
print(red.dims, red.shape, float(red.max()))

In [ ]:
after_dn = rxr.open_rasterio(after_path, masked=True)
after_dn = after_dn.assign_coords(band=list(after_dn.attrs["long_name"]))
print("NaN cells per band, after image:")
print(after_dn.isnull().sum(dim=["y", "x"]).to_series())

The DataArray has the dimensions `('band', 'y', 'x')` and the shape (10, 1006, 702). With `masked=True` its dtype is `float32`. After `assign_coords`, the band coordinates are the names B02 to B12. `.sel(band="B04")` returns a 2-D DataArray of the red band; its highest stored value is 6,904.

`.isnull().sum(dim=["y", "x"])` counts the `NaN` cells of each band, adding up over rows and columns. The after image has a handful of 0 cells in the visible bands: 12 in blue, 25 in green and 125 in red, plus 3 in B08. They lie in deep shadow, where the corrected value fell to 0. Treat them as nodata, as `masked=True` did.

`.to_series()` turns the result into a pandas Series (A9) so it prints as a neat list.

### Practice

**Q1 · one line:** Select the NIR band B08 of `before_dn` by name and print its mean, wrapped in `float()` and rounded to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float(before_dn.sel(band="B08").mean()), 1)
```

About 3,002.7. That is a stored number, not a reflectance yet: step 4 converts it.
</details>

**Q2 · two lines:** Select the two SWIR bands of `after_dn` in one call, with a list of names, and print the shape and the band coordinates of the result.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
swir = after_dn.sel(band=["B11", "B12"])
print(swir.shape, swir.band.values)
```

(2, 1006, 702) with the bands `['B11' 'B12']`: a list of names keeps the band dimension, one name drops it.
</details>

**Q3 · three lines:** Use `.isel(band=6)` (selection by **position**, C4) on `before_dn`, and check that it holds the same values as `.sel(band="B08")`. Print the band coordinate of the `.isel` result and the result of `.equals()`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
by_position = before_dn.isel(band=6)
print(by_position.band.values)
print(by_position.equals(before_dn.sel(band="B08")))
```

Position 6 (the 7th band) is B08, and `.equals` gives `True`. Names are safer: `.isel(band=7)` would quietly give B8A instead.
</details>

## 4. From stored numbers to reflectance

Images are stored as whole numbers because they take less space than decimals. These stored values are often called **digital numbers** (DN). The file's scale tells you how to turn them into reflectance:

**reflectance = DN × 0.0001** (the offset is 0).

So a DN of 2,914 means a reflectance of 0.2914: the ground reflected about 29% of the light in that band.

A warning for your own downloads: from January 2022, ESA added an **offset of −0.1** to Sentinel-2 Level-2A data (stored values got 1,000 added). Our two images are older, so their offset is 0, as the file says. Always read `src.offsets` before you scale.

Once the values are reflectances, statistics per band become meaningful. `.median(dim=["y", "x"])` gives one median per band.

In [ ]:
before = before_dn * 0.0001         # reflectance, 0 to about 1
after = after_dn * 0.0001
print(before.dtype, before.rio.crs)
print("cells with a B08 reflectance above 1:", int((before.sel(band="B08") > 1).sum()))

stats = pd.DataFrame({
    "min": before.min(dim=["y", "x"]).values,
    "median": before.median(dim=["y", "x"]).values,
    "p98": before.quantile(0.98, dim=["y", "x"]).values,
    "max": before.max(dim=["y", "x"]).values,
}, index=before.band.values)
stats.round(4)

In [ ]:
left, bottom, right, top = before.rio.bounds()
extent_km = [left / 1000, right / 1000, bottom / 1000, top / 1000]   # as in C4

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
for ax, name, label in zip(axes, ["B04", "B08"], ["red (B04)", "near-infrared (B08)"]):
    band = before.sel(band=name).values
    low, high = np.nanpercentile(band, [2, 98])            # percentile stretch, C2
    im = ax.imshow(band, extent=extent_km, cmap="gray", vmin=low, vmax=high)
    fig.colorbar(im, ax=ax, label="Reflectance", shrink=0.6, extend="both")
    ax.set_title(f"{label}, 29 October 2020")
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
plt.show()

Multiplying by 0.0001 keeps the CRS and the coordinates, so `before` is still a map.

The visible bands are dark: the median reflectance is about 0.018 in blue, 0.044 in green and 0.030 in red. The NIR bands are much brighter, with medians of about 0.29 to 0.30. That is the signature of a green landscape: plants absorb most blue and red light, and reflect much NIR.

71 cells have a B08 reflectance above 1, up to about 1.52. A likely cause is steep slopes facing the sun: surface reflectance is calculated as if the ground were flat, and a slope tilted towards the sun receives more light than flat ground.

The two grey maps show the same place in two bands. In **red**, forest is dark and the river bed bright. In **NIR**, forest is bright, the river is a dark line, and the slopes in shadow stand out in black. Each map has its own 2–98% stretch (C2).

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** What fraction of the light did the median cell reflect in the SWIR 1 band? Look up `"B11"` in the `median` column of `stats` with `.loc` (A9).

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
stats.loc["B11", "median"]
```

About 0.179, roughly 18% of the light: less than NIR, more than the visible bands.
</details>

**Q2 · two lines:** rioxarray can scale for you: open the after image with `mask_and_scale=True`, then print the dtype and the median of band 3 (red), wrapped in `float()` and rounded to 4 decimals.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
auto = rxr.open_rasterio(after_path, mask_and_scale=True)
print(auto.dtype, round(float(auto.sel(band=3).median()), 4))
```

`float32` and about 0.0271, the same as `after`'s red median in the example. `mask_and_scale=True` turns nodata into `NaN` **and** applies the file's scale and offset. Here the scale is in the file; for files without one, multiply by hand, as in the example.
</details>

**Q3 · three lines:** Make the same statistics table for the **after** image, but only the `median` column, and print the change from before: `after median − before median` for each band, rounded to 4 decimals.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
after_median = after.median(dim=["y", "x"]).values
change = pd.Series(after_median - stats["median"].values, index=stats.index)
print(change.round(4))
```

Blue rose by about 0.004, while red and the NIR bands fell (NIR by about 0.05). Some of this is real change on the ground, but a different date, sun angle, atmosphere and satellite also shift whole images. D3 returns to this.
</details>

## 5. True colour: three bands and a stretch

A screen mixes **red, green and blue** light into every colour. To see a satellite image as our eyes would, put the red band (B04) into the screen's red channel, green (B03) into green, and blue (B02) into blue. This is a **true-colour composite**.

`imshow` draws a 3-D array of shape **(rows, columns, 3)** as colours, with values from 0 to 1. `np.dstack([red, green, blue])` stacks three 2-D arrays along a third axis in that shape ("d" for depth).

Reflectances in the visible bands are small, mostly below 0.15. Shown as they are, almost everything is near 0, so the picture is **dark**. The fix is a **stretch**, as in C2: divide each band by a high value, such as its **98th percentile**, so that it fills the range 0 to 1. Then `np.clip(rgb, 0, 1)` raises anything below 0 to 0 and lowers anything above 1 to 1.

**Predict:** in the unstretched picture, which colours will you see?

In [ ]:
red = before.sel(band="B04").values
green = before.sel(band="B03").values
blue = before.sel(band="B02").values

rgb_raw = np.dstack([red, green, blue])
print(rgb_raw.shape, "| highest value:", round(float(np.nanmax(rgb_raw)), 3))

tops = np.nanpercentile(rgb_raw, 98, axis=(0, 1))      # 98th percentile of each band
print("98th percentiles (red, green, blue):", tops.round(4))
rgb = np.clip(rgb_raw / tops, 0, 1)                    # each band divided by its own top

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8))
axes[0].imshow(np.clip(rgb_raw, 0, 1), extent=extent_km)
axes[0].set_title("True colour, no stretch")
axes[1].imshow(rgb, extent=extent_km)
axes[1].set_title("True colour, 98th-percentile stretch")
for ax in axes:
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.suptitle("Melamchi valley, Sentinel-2, 29 October 2020")
plt.show()

`rgb_raw` has the shape (1006, 702, 3): rows, columns and three colours. Its highest value is about 0.69, but the 98th percentiles are only about 0.136 (red), 0.121 (green) and 0.080 (blue).

Without a stretch, the picture is almost black: the valley shows only as faint lines. With the stretch, it looks like a photograph: dark green forest, paler fields and the bright, braided river bed running north to south.

`np.nanpercentile(rgb_raw, 98, axis=(0, 1))` works out one percentile per colour, over all rows and columns. Dividing the (1006, 702, 3) array by those three numbers divides each colour by its own top: NumPy **broadcasting** (A8). Cells brighter than their band's 98th percentile become 1, pure white, such as the river gravel. That is the price of a stretch.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** What share of the red values of `rgb` became exactly 1 after clipping? Use `rgb[:, :, 0]` and print the percentage rounded to 1 decimal.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(float((rgb[:, :, 0] == 1).mean() * 100), 1)
```

About 2.0%, as you would expect: the 2% of cells above the 98th percentile all become 1.
</details>

**Q2 · two lines:** Make a stretch with the **99.5th** percentile instead, and draw it. Use `tops_995` as the name of the new tops.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
tops_995 = np.nanpercentile(rgb_raw, 99.5, axis=(0, 1))
fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(np.clip(rgb_raw / tops_995, 0, 1), extent=extent_km)
ax.set_title("True colour, 99.5th-percentile stretch, 29 October 2020")
plt.show()
```

The picture is darker: the tops are higher, so most cells get smaller values. The bright river bed now shows more detail. There is no single correct stretch; choose the one that shows what you need, and say which one you used.
</details>

**Q3 · three lines:** Draw the true colour of the **after** image with its own 98th-percentile stretch. Build `rgb_after_raw` with `np.dstack` from the B04, B03 and B02 bands of `after`, as in the example.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
rgb_after_raw = np.dstack([after.sel(band=b).values for b in ["B04", "B03", "B02"]])
rgb_after = np.clip(np.nan_to_num(rgb_after_raw / np.nanpercentile(rgb_after_raw, 98, axis=(0, 1))), 0, 1)
fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(rgb_after, extent=extent_km)
ax.set_title("True colour, 11 November 2021")
plt.show()
```

A wide, bright white band now runs down the valley: the flood sediment. `np.nan_to_num` turns the few `NaN` cells of the after image into 0 (black), because `imshow` cannot colour `NaN` in a colour picture. The list comprehension (A3) builds the three bands in one line.
</details>

## 6. False colour: seeing the invisible

Nothing forces you to show red light in the red channel. A **false-colour composite** puts invisible bands on the screen's colours.

- **NIR – red – green** (B08, B04, B03): healthy vegetation reflects NIR strongly, so it appears **red**. Bare ground, gravel and buildings look grey to cyan. This is the classic colour-infrared view.
- **SWIR 2 – NIR – red** (B12, B08, B04): vegetation appears **green**, bare sediment and dry rock appear **pink to magenta** (high SWIR), and water is very dark, because water absorbs NIR and SWIR.

The example wraps the steps of step 5 in a function (A5), so that any three bands become a picture in one line. `tops` is optional: without it, each band gets its own 98th percentile.

In [ ]:
def composite(image, band_names, tops=None):
    """Three bands as an RGB picture (rows, columns, 3), each divided by its top value and clipped to 0-1."""
    rgb = np.dstack([image.sel(band=name).values for name in band_names])
    if tops is None:
        tops = np.nanpercentile(rgb, 98, axis=(0, 1))     # the 98th percentile of each band
    return np.clip(np.nan_to_num(rgb / tops), 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 8))
axes[0].imshow(composite(before, ["B08", "B04", "B03"]), extent=extent_km)
axes[0].set_title("NIR-red-green (B08, B04, B03)")
axes[1].imshow(composite(before, ["B12", "B08", "B04"]), extent=extent_km)
axes[1].set_title("SWIR-NIR-red (B12, B08, B04)")
for ax in axes:
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.suptitle("False colour, Sentinel-2, 29 October 2020")
plt.show()

In the NIR–red–green picture, most of the valley is **red**: forest and green fields. The river bed, roads and bare patches are pale cyan. Shadowed slopes look dark red to black.

In the SWIR–NIR–red picture, the same vegetation is **green**, and bare or dry ground turns **pink**: the gravel of the river bed, bare fields and a few bright patches on the hillsides. The river channel itself is a thin dark blue to purple line: water absorbs NIR and SWIR. It is so narrow that few cells hold only water (step 7).

The same function, three different band lists: once the steps are in a function, making a composite takes one line.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** Draw the after image as a NIR–red–green composite with `composite`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(composite(after, ["B08", "B04", "B03"]), extent=extent_km)
ax.set_title("NIR-red-green, 11 November 2021")
plt.show()
```

Red vegetation on the slopes, and a wide pale cyan strip along the river where the flood left sediment.
</details>

**Q2 · two lines:** Make a SWIR composite with the **other** SWIR band, B11 – B08 – B04, for the after image. Store the picture in `swir1_after`, print its shape and draw it.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
swir1_after = composite(after, ["B11", "B08", "B04"])
print(swir1_after.shape)
fig, ax = plt.subplots(figsize=(6, 8))
ax.imshow(swir1_after, extent=extent_km)
ax.set_title("SWIR 1-NIR-red, 11 November 2021")
plt.show()
```

(1006, 702, 3). The sediment strip shows pink to purple, softer than with B12, and vegetation looks yellow-green. Both SWIR bands separate bare ground from plants.
</details>

**Q3 · three lines:** Check the function: `composite(before, ["B04", "B03", "B02"])` should equal `rgb` from step 5. Compare them with `np.allclose` (C4 compared transforms with a tolerance in the same spirit), and print the minimum and maximum of the composite.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
tc = composite(before, ["B04", "B03", "B02"])
print(np.allclose(tc, rgb))
print(tc.min(), tc.max())
```

`True`, and the values run from about 0.0007 to 1.0. A function that reproduces a result you trust can be trusted on new inputs.
</details>

## 7. Spectral profiles: the fingerprint of a surface

The reflectance of one cell in every band, drawn against wavelength, is its **spectral profile** (or spectral signature). Different surfaces have different profiles, which is why bands can tell them apart.

The example reads four cells with `.sel(x=..., y=..., method="nearest")`, as in C7, and plots their 10 reflectances against the bands' central wavelengths.

**How these cells were chosen.** I looked at the composites of steps 5 and 6 and picked four places by eye, then checked their numbers:

- **forest**: a dark green slope in the north-east (dark red in false colour), with NIR far above red;
- **built-up area**: grey roofs where the two rivers meet in the south-east, with a flat, medium-bright profile;
- **river**: the darkest river cell in the south, with NIR below green, the sign of water. The river is only one or two cells wide here, so this cell mixes water with gravel (review 2 checks this);
- **flood sediment**: a cell in the bright strip of the after image, where the flood left sediment.

The first three come from the before image, the sediment cell from the after image.

In [ ]:
wavelength_nm = bands["centre_nm"].values
places = {                                   # name: (image, easting m, northing m)
    "forest (2020)": (before, 361010, 3089990),
    "built-up area (2020)": (before, 359610, 3079190),
    "river, water and gravel (2020)": (before, 359710, 3076730),
    "flood sediment (2021)": (after, 359610, 3077990),
}

profiles = {}
for name, (image, x_m, y_m) in places.items():
    profiles[name] = image.sel(x=x_m, y=y_m, method="nearest").values
profiles = pd.DataFrame(profiles, index=bands["band"])
profiles.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name in profiles.columns:
    ax.plot(wavelength_nm, profiles[name], marker="o", label=name)
ax.axvspan(400, 700, color="gold", alpha=0.2)
ax.set_xlabel("Wavelength (nm)")
ax.set_ylabel("Reflectance")
ax.set_title("Spectral profiles of four cells, Sentinel-2")
ax.legend()
plt.show()

Each column of the table is one profile. Read them like fingerprints:

- **Forest** is dark in the visible (red only 0.032), with a small bump at green (0.057), the reason leaves look green. From red to NIR its reflectance jumps to about 0.41: the **red edge**. It falls again in SWIR (0.216 and 0.098), where water inside the leaves absorbs light.
- **Built-up area** is flat and medium-bright everywhere, from about 0.16 to 0.28: roofs, roads and bare ground mixed in one 20 m cell.
- **River** is low in every band. It peaks at green (0.041) and drops to 0.020 in NIR, like water, but its SWIR (0.029) is not the lowest value, as it would be for pure water: gravel in the cell adds some.
- **Flood sediment** is bright in every band, from about 0.24 in blue to 0.47 in SWIR 2. There is no red edge: nothing green grows on fresh sediment.

The yellow strip marks visible light. Notice how little of each profile it covers: most of the information is in the infrared.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** Which band has the highest reflectance in the forest profile? Use `.idxmax()` on the column (it returns the index label of the largest value).

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
profiles["forest (2020)"].idxmax()
```

`'B8A'`, the narrow NIR band (0.434), closely followed by B08 (0.413). Forest is brightest in the near-infrared.
</details>

**Q2 · two lines:** How much brighter is the forest in NIR than in red? Divide its B08 reflectance by its B04 reflectance and round to 1 decimal. Do the same for the flood sediment.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(profiles.loc["B08", "forest (2020)"] / profiles.loc["B04", "forest (2020)"], 1))
print(round(profiles.loc["B08", "flood sediment (2021)"] / profiles.loc["B04", "flood sediment (2021)"], 1))
```

About 12.9 times for forest, but only about 1.1 times for sediment. This ratio of NIR to red is the idea behind the vegetation indices of D2.
</details>

**Q3 · three lines:** What did the sediment cell look like **before** the flood? Read the same cell (359610, 3077990) from `before`, add it to `profiles` as a new column `"sediment cell (2020)"`, and print the B04, B08 and B12 rows of the two sediment-cell columns.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
profiles["sediment cell (2020)"] = before.sel(x=359610, y=3077990, method="nearest").values
cols = ["sediment cell (2020)", "flood sediment (2021)"]
print(profiles.loc[["B04", "B08", "B12"], cols].round(3))
```

In 2020 the cell had a low red (about 0.078) and a high NIR (about 0.294): something green grew there. In 2021 red is about 0.296 and NIR about 0.312: the red edge is gone. The flood replaced plants with sediment. D3 maps such changes for the whole valley.
</details>

## 8. Before and after: a first look at the flood

To compare two dates by eye, both pictures need the **same stretch**. If each image is stretched by its own percentiles, a brighter image is pulled down and a darker one pushed up, and real differences can vanish.

The example takes the 98th percentiles of the **before** image and uses them as `tops` for both dates. Equal colours then mean equal reflectances.

It also marks Melamchi. Wikipedia gives the municipality's centre as 27°50′N 85°34′E, about 27.83° N and 85.56° E. That position is rounded to whole minutes of arc, so it is only good to a kilometre or two. The point is converted to UTM 45N with `to_crs`, as in B4.

Titles name the dates, and the credit line goes on the figure.

In [ ]:
true_colour = ["B04", "B03", "B02"]
tops_before = np.nanpercentile(np.dstack([before.sel(band=b).values for b in true_colour]), 98, axis=(0, 1))

melamchi = gpd.GeoSeries(gpd.points_from_xy([85.56], [27.83]), crs="EPSG:4326").to_crs(32645).iloc[0]
print("Melamchi (approx.):", round(melamchi.x), round(melamchi.y))

fig, axes = plt.subplots(1, 2, figsize=(11, 7.6))
for ax, image, title in zip(axes, [before, after], ["Before: 29 October 2020", "After: 11 November 2021"]):
    ax.imshow(composite(image, true_colour, tops=tops_before), extent=extent_km)
    ax.plot(melamchi.x / 1000, melamchi.y / 1000, marker="+", color="red", markersize=14, markeredgewidth=2)
    ax.annotate("Melamchi (approx.)", (melamchi.x / 1000, melamchi.y / 1000), xytext=(-10, 6), textcoords="offset points", ha="right", color="white")
    ax.set_title(title)
    ax.set_xlabel("Easting (km)")
axes[0].set_ylabel("Northing (km)")
fig.suptitle("Melamchi valley, true colour, same stretch for both dates")
fig.subplots_adjust(top=0.92, bottom=0.08)
fig.text(0.01, 0.01, "Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.", fontsize=9)
plt.show()

for name in true_colour:
    print(name, "median before:", round(float(before.sel(band=name).median()), 4), "| after:", round(float(after.sel(band=name).median()), 4))

In October 2020 the Melamchi river is a narrow, braided channel between green fields and forest. In November 2021 a **wide, bright strip** of sediment fills the valley floor, from the northern edge of the image down past the place where the two rivers meet, and on to the south. Fields and trees that lined the river are gone under it.

The Melamchi mark lands on the hillside west of the river, about 358 km east and 3,079 km north. The built-up area you saw in step 7 lies about 1.4 km further east, by the river. Remember that the mark is rounded.

The whole after image also looks a little **bluer and hazier**. The median blue reflectance rose from about 0.018 to about 0.022, while red fell slightly. Not all of that is change on the ground: the two images were taken by different satellites (2A and 2B), 13 days apart in the season, through a different atmosphere and with a lower sun. Such differences cause **false change**, which D3 has to separate from the flood.

*Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.*

### Practice

**Q1 · one line:** How far is the Melamchi mark from the built-up cell of step 7 (359610, 3079190)? Use B2's `.distance` with a shapely `Point` in UTM metres, and print it in km rounded to 2 decimals.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
from shapely.geometry import Point
round(melamchi.distance(Point(359610, 3079190)) / 1000, 2)
```

About 1.42 km. Coordinates rounded to whole minutes can be off by that much, so treat the mark as "near here", not as a precise location.
</details>

**Q2 · two lines:** Zoom in: cut a 3 km × 4 km box around the place where the two rivers meet from both images with `.rio.clip_box(358000, 3076500, 361000, 3080500)` (C3), and print the shape of each clip.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
zoom_before = before.rio.clip_box(358000, 3076500, 361000, 3080500)
zoom_after = after.rio.clip_box(358000, 3076500, 361000, 3080500)
print(zoom_before.shape, zoom_after.shape)
```

Both clips have the shape (10, 200, 150): 10 bands, 200 rows × 150 columns of 20 m cells. `clip_box` works on all bands at once.
</details>

**Q3 · three lines:** Draw `zoom_before` and `zoom_after` side by side in true colour with `tops_before`. Give each map `extent=` from its own `rio.bounds()` in km and a title with its date.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(10, 6))
for ax, image, title in zip(axes, [zoom_before, zoom_after], ["29 October 2020", "11 November 2021"]):
    l, b, r, t = image.rio.bounds()
    ax.imshow(composite(image, true_colour, tops=tops_before), extent=[l / 1000, r / 1000, b / 1000, t / 1000])
    ax.set_title(title)
plt.show()
```

Close up, the 2021 sediment covers the channel and the fields beside it, up to the edge of the built-up area where the rivers meet. Each clip needs its own extent, because its bounds differ from the full image.
</details>

## 9. When it breaks: band 0 does not exist

You know from A3 that Python counts from 0, so you ask rasterio for the first band with `read(0)`. Run the cell.

In [ ]:
with rasterio.open(before_path) as src:
    first_band = src.read(0)

The last line of the error says:

`IndexError: band index 0 out of range (not in (1, 2, 3, 4, 5, 6, 7, 8, 9, 10))`

- **IndexError**: you asked for a position that does not exist, as with a list in A3.
- **band index 0 out of range**: rasterio numbers bands from **1**, like the GDAL library underneath it. The message even lists the valid numbers, 1 to 10.

So there are two ways of counting in this lesson:

| Where | First band | B08 |
| --- | --- | --- |
| `src.read(...)`, rasterio | 1 | 8th band → `read(7)` |
| a NumPy array from `read()` | 0 | `all_dn[6]` |
| rioxarray before naming, `.sel(band=...)` | 1 | `.sel(band=7)` |
| rioxarray after naming | `"B02"` | `.sel(band="B08")` |

Band numbers are confusing; names are not. That is why the lesson names the bands as soon as it opens a file.

**Try it:** fix the cell so that it reads the blue band, B02.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · A band table for both dates

Build a pandas table with one row per band and four columns: the band's central wavelength (from `bands`), the median reflectance before, the median after, and the change in percent of the before value. Round to 3 decimals and print it. Which bands changed most?

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
table = pd.DataFrame({
    "centre_nm": bands["centre_nm"].values,
    "before": before.median(dim=["y", "x"]).values.astype(float),    # float64 prints neatly (A8)
    "after": after.median(dim=["y", "x"]).values.astype(float),
}, index=bands["band"])
table["change_pct"] = (table["after"] - table["before"]) / table["before"] * 100
print(table.round(3))
```

Blue rose by about 23%, while every other band fell, from about 10% (red) to about 18% (B06). Most of the valley is not flooded, so these whole-image changes come mostly from the different dates, sun angles and satellites, not from the flood. A whole-image median cannot see a narrow strip of sediment.
</details>

### Review 2 · Is there any clear water?

A cell of clear water reflects more green than NIR **and** more green than SWIR 1. Make a mask of such cells in the before image. Then use `ndimage.minimum_filter(mask, size=3)` from C6 to find cells whose whole 3 × 3 window is water. Print both counts, and do the same for the after image.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
from scipy import ndimage

for name, image in [("before", before), ("after", after)]:
    green = image.sel(band="B03").values
    water = (green > image.sel(band="B08").values) & (green > image.sel(band="B11").values)
    solid = ndimage.minimum_filter(water.astype(int), size=3) == 1
    print(name, "water-like cells:", water.sum(), "| with all 8 neighbours water-like:", solid.sum())
```

241 water-like cells before and 935 after, but **0** whole 3 × 3 blocks on either date. The river is never 60 m of clear water across, so every "water" cell also holds some gravel or bank. Such **mixed pixels** are why the river profile of step 7 is not a textbook water curve. `minimum_filter` on 0s and 1s gives 1 only where every cell of the window is 1 (C6). `NaN > number` is `False`, so the few `NaN` cells count as not water.
</details>

### Review 3 · A SWIR map for a report

Draw the after image as a SWIR 2–NIR–red composite with the **before** image's 98th percentiles for these three bands as `tops`. Mark Melamchi, label the axes in km, give a title with the date, add the credit line with `fig.text`, and save it as `d1_swir_after.png` in `OUT` at 200 dpi.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
swir_bands = ["B12", "B08", "B04"]
swir_tops = np.nanpercentile(np.dstack([before.sel(band=b).values for b in swir_bands]), 98, axis=(0, 1))

fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(composite(after, swir_bands, tops=swir_tops), extent=extent_km)
ax.plot(melamchi.x / 1000, melamchi.y / 1000, marker="+", color="yellow", markersize=14, markeredgewidth=2)
ax.set_title("SWIR 2-NIR-red (B12, B08, B04), 11 November 2021")
ax.set_xlabel("Easting (km)")
ax.set_ylabel("Northing (km)")
fig.text(0.01, 0.01, "Contains modified Copernicus Sentinel data 2020–2021, processed by ESA.", fontsize=8)
fig.savefig(OUT / "d1_swir_after.png", dpi=200, bbox_inches="tight")
plt.show()
```

The sediment strip glows pink to magenta against green vegetation along the whole valley. Dry, bare fields on the slopes are pink too: colour alone does not separate flood sediment from other bare ground. Using the before image's tops for the after image keeps the colours comparable with a before map drawn the same way. `fig.text` puts the credit in figure coordinates, so it travels with the saved file.
</details>

### Review 4 · Profiles along the valley

Read the reflectance of the after image in the SWIR 2 band (B12) and in NIR (B08) at five points along the valley floor, all at easting 357,000 m and northings 3,082,000 to 3,090,000 m every 2,000 m (C7's `.sel` with paired `xr.DataArray` coordinates). Put them in a pandas table and add a column with B12 divided by B08.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
import xarray as xr

northings_m = np.arange(3082000, 3090001, 2000)
xs = xr.DataArray([357000] * len(northings_m), dims="point")
ys = xr.DataArray(northings_m, dims="point")

at_points = after.sel(band=["B08", "B12"]).sel(x=xs, y=ys, method="nearest")
valley = pd.DataFrame({"northing_m": northings_m, "B08": at_points.sel(band="B08").values, "B12": at_points.sel(band="B12").values})
valley["B12_over_B08"] = valley["B12"] / valley["B08"]
print(valley.round(3))
```

Only the point at northing 3,088,000 m lies on bright sediment: B12 is higher than B08 there, a ratio of about 1.4. The other four have ratios from about 0.27 to 0.74, the values of vegetation and mixed ground, because the winding river is not always at easting 357,000 m. A straight line of points samples a winding valley unevenly, which is why D3 works with whole maps instead.
</details>

## Recap

- A passive optical satellite measures **reflected sunlight** in **bands** of wavelength: visible (about 400–700 nm), near-infrared and short-wave infrared. **Reflectance** is the fraction of light reflected, from 0 to 1.
- **Sentinel-2** (2A and 2B in 2020–2021) images each place about every 5 days in 13 bands at 10, 20 and 60 m. **Level-2A** gives surface reflectance. Our files hold 10 bands on one 20 m grid in UTM 45N.
- rasterio: `src.count`, `src.descriptions`, `src.scales`, `src.offsets`; `read(1)` is one band, `read()` all (bands, rows, columns); bands count from **1**. rioxarray: name the bands with `assign_coords(band=...)` and pick them with `.sel(band="B04")`.
- **Reflectance = stored value × 0.0001** for these images (offset 0; newer Sentinel-2 data have an offset). Visible bands are dark, so composites need a **stretch**: divide by the 98th percentile and clip to 0–1.
- **True colour** (B04, B03, B02) shows the land as our eyes would; **false colour** (B08, B04, B03 or B12, B08, B04) shows vegetation and bare sediment clearly. **Spectral profiles** are fingerprints of surfaces. Compare dates with the **same stretch**, and expect some false change between images.

### Words you learned

| Word | Meaning |
| --- | --- |
| wavelength | The length of one wave of light, here in nanometres (nm) |
| nanometre (nm) | One billionth of a metre; 1,000 nm = 1 µm |
| electromagnetic spectrum | All wavelengths of light and radiation, from radio waves to gamma rays |
| visible / NIR / SWIR | Light our eyes see (about 400–700 nm) / near-infrared / short-wave infrared |
| reflectance | The fraction of incoming light that a surface reflects, 0 to 1 |
| passive / active sensor | Measures reflected sunlight / sends its own signal, such as radar |
| band | One layer of an image, measuring one range of wavelengths |
| central wavelength | The middle of the range of wavelengths a band measures |
| Sentinel-2 | The Copernicus optical satellite mission run by ESA |
| Level-2A | Sentinel-2 data corrected for the atmosphere: surface reflectance |
| digital number (DN) | The whole number stored in an image file, before scaling |
| scale factor / offset | Numbers that turn stored values into real ones: value × scale + offset |
| STAC catalogue | An online, searchable index of satellite scenes |
| composite | A colour picture made from three bands |
| true colour / false colour | Red, green and blue bands on the screen's red, green and blue / any other bands |
| stretch | Rescaling values so that they fill the range of colours |
| `np.dstack` | Stacks 2-D arrays into one 3-D array along a third axis |
| red edge | The steep rise in plant reflectance between red and NIR |
| spectral profile | The reflectance of one cell in every band, plotted against wavelength |
| mixed pixel | A cell that contains more than one kind of surface |
| false change | A difference between two images that is not a change on the ground |

## Next

**D2 · Spectral indices:** combine bands into single numbers such as NDVI that measure vegetation, water and bare ground, and avoid the integer trap of `uint16` band maths.

<!-- pygile-nav -->
**PyGILE Course** · lesson 30 of 32 · [Course home](../README.md)  
← Previous: [C8 · Interpolation: a rainfall map from 24 gauges](../C_Raster/C08_Interpolation.ipynb) · Next: [D2 · Spectral indices: vegetation, water and bare ground in the Melamchi valley](D02_Spectral_indices.ipynb) →